# CMS Medicare Claims Data Cleaning

This notebook performs data cleaning and transformation on the CMS DE-SynPUF dataset.

## Objectives:
1. Load and explore raw CMS data
2. Handle missing values
3. Remove duplicates
4. Normalize categorical fields (ICD/CPT codes)
5. Standardize date formats
6. Save cleaned data for further processing


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
project_root = Path.cwd().parent
raw_data_path = project_root / 'data' / 'raw' / 'cms-synpuf' / 'sample_1'
processed_data_path = project_root / 'data' / 'processed' / 'cms-synpuf'

# Create processed directory if it doesn't exist
processed_data_path.mkdir(parents=True, exist_ok=True)

print(f"Raw data path: {raw_data_path}")
print(f"Processed data path: {processed_data_path}")
print(f"Raw data exists: {raw_data_path.exists()}")


## 1. Beneficiary Data Cleaning


In [ ]:
def clean_beneficiary_data():
    """Clean and process beneficiary data for all years"""
    
    beneficiary_files = [
        'DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv',
        'DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv',
        'DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv'
    ]
    
    cleaned_beneficiaries = []
    
    for file in beneficiary_files:
        year = file.split('_')[2]  # Extract year from filename
        file_path = raw_data_path / 'beneficiary' / file
        
        if not file_path.exists():
            print(f"Warning: {file} not found at {file_path}")
            continue
            
        print(f"Processing {file}...")
        
        # Load data
        df = pd.read_csv(file_path, low_memory=False)
        print(f"  Original shape: {df.shape}")
        
        # Add year column
        df['year'] = int(year)
        
        # Handle missing values in key demographic fields
        # Replace 'U' (Unknown) with NaN for better analysis
        demographic_cols = ['BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND']
        for col in demographic_cols:
            if col in df.columns:
                df[col] = df[col].replace('U', np.nan)
        
        # Standardize date formats
        date_cols = ['BENE_BIRTH_DT', 'BENE_DEATH_DT', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 
                    'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 
                    'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA']
        
        for col in date_cols:
            if col in df.columns:
                # Convert to datetime, errors='coerce' will set invalid dates to NaT
                df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')
        
        # Remove duplicates based on DESYNPUF_ID (patient ID)
        initial_count = len(df)
        df = df.drop_duplicates(subset=['DESYNPUF_ID'], keep='first')
        duplicates_removed = initial_count - len(df)
        print(f"  Removed {duplicates_removed} duplicate patients")
        
        # Data quality checks
        print(f"  Missing values in key columns:")
        key_cols = ['DESYNPUF_ID', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD']
        for col in key_cols:
            if col in df.columns:
                missing_pct = (df[col].isna().sum() / len(df)) * 100
                print(f"    {col}: {missing_pct:.1f}%")
        
        cleaned_beneficiaries.append(df)
        print(f"  Final shape: {df.shape}\n")
    
    if not cleaned_beneficiaries:
        print("No beneficiary files found!")
        return None
    
    # Combine all years
    combined_df = pd.concat(cleaned_beneficiaries, ignore_index=True)
    
    # Save cleaned data
    output_path = processed_data_path / 'beneficiary_cleaned_all_years.csv'
    combined_df.to_csv(output_path, index=False)
    print(f"Saved cleaned beneficiary data: {output_path}")
    print(f"Total records: {len(combined_df)}")
    
    return combined_df

# Run beneficiary cleaning
beneficiary_df = clean_beneficiary_data()


## 2. Inpatient Claims Data Cleaning


In [ ]:
def clean_inpatient_data():
    """Clean and process inpatient claims data"""
    
    file_path = raw_data_path / 'inpatient' / 'DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv'
    
    if not file_path.exists():
        print(f"Warning: Inpatient file not found at {file_path}")
        return None
    
    print(f"Processing inpatient claims data...")
    
    # Load data
    df = pd.read_csv(file_path, low_memory=False)
    print(f"Original shape: {df.shape}")
    
    # Standardize date formats
    date_cols = ['CLM_FROM_DT', 'CLM_THRU_DT', 'NCH_WKLY_PROC_DT']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')
    
    # Clean diagnosis codes (ICD-9)
    diag_cols = [col for col in df.columns if col.startswith('ICD9_DGNS_CD')]
    for col in diag_cols:
        if col in df.columns:
            # Remove leading/trailing spaces and convert to uppercase
            df[col] = df[col].astype(str).str.strip().str.upper()
            # Replace 'nan' strings with actual NaN
            df[col] = df[col].replace('NAN', np.nan)
    
    # Clean procedure codes (ICD-9)
    proc_cols = [col for col in df.columns if col.startswith('ICD9_PRCDR_CD')]
    for col in proc_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
            df[col] = df[col].replace('NAN', np.nan)
    
    # Handle missing values in key fields
    key_cols = ['DESYNPUF_ID', 'CLM_ID', 'CLM_FROM_DT', 'CLM_THRU_DT']
    for col in key_cols:
        if col in df.columns:
            missing_count = df[col].isna().sum()
            if missing_count > 0:
                print(f"  Warning: {missing_count} missing values in {col}")
    
    # Remove duplicates
    initial_count = len(df)
    df = df.drop_duplicates(subset=['CLM_ID'], keep='first')
    duplicates_removed = initial_count - len(df)
    print(f"Removed {duplicates_removed} duplicate claims")
    
    # Data quality summary
    print(f"\nData quality summary:")
    print(f"  Total claims: {len(df)}")
    print(f"  Unique patients: {df['DESYNPUF_ID'].nunique()}")
    if 'CLM_FROM_DT' in df.columns:
        print(f"  Date range: {df['CLM_FROM_DT'].min()} to {df['CLM_FROM_DT'].max()}")
    
    # Save cleaned data
    output_path = processed_data_path / 'inpatient_claims_cleaned.csv'
    df.to_csv(output_path, index=False)
    print(f"\nSaved cleaned inpatient data: {output_path}")
    
    return df

# Run inpatient cleaning
inpatient_df = clean_inpatient_data()


## 3. Data Quality Summary


In [ ]:
def generate_data_summary():
    """Generate a summary of all cleaned datasets"""
    
    print("=" * 60)
    print("CMS DE-SynPUF DATA CLEANING SUMMARY")
    print("=" * 60)
    
    datasets = {
        'Beneficiary': beneficiary_df,
        'Inpatient Claims': inpatient_df
    }
    
    for name, df in datasets.items():
        if df is not None:
            print(f"\n{name}:")
            print(f"  Records: {len(df):,}")
            print(f"  Columns: {len(df.columns)}")
            print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
            
            if 'DESYNPUF_ID' in df.columns:
                print(f"  Unique patients: {df['DESYNPUF_ID'].nunique():,}")
            
            if 'CLM_FROM_DT' in df.columns:
                print(f"  Date range: {df['CLM_FROM_DT'].min().date()} to {df['CLM_FROM_DT'].max().date()}")
        else:
            print(f"\n{name}: Not processed (file not found)")
    
    print("\n" + "=" * 60)
    print("CLEANED DATA FILES SAVED TO:")
    print(f"{processed_data_path}")
    print("=" * 60)

# Generate summary
generate_data_summary()
